<a href="https://colab.research.google.com/github/KrishnaTulasiSatti/Ai-PortFolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [5]:
# Create a sample text file with 3 resumes separated by '---'
content = """Name: Alice Johnson
Email: alice.j@example.com
Phone: 555-0101
Education: BS in Computer Science, State University, 2022
Skills: Python, SQL, Machine Learning
Projects: Built an automated data pipeline for sentiment analysis
Experience Years: 2.0
---
Name: Bob Smith
Email: bob.s@example.com
Phone: 555-0102
Education: BS in Mechanical Engineering, Tech Institute, 2020
Skills: AutoCAD, SolidWorks, MATLAB
Projects: Designed a robotic arm for assembly line tasks
Experience Years: 4.5
---
Name: Charlie Davis
Email: charlie.d@example.com
Phone: 555-0103
Education: BA in Marketing, City College, 2023
Skills: SEO, Google Analytics, Content Strategy
Projects: Managed a social media campaign increasing leads by 30%
Experience Years: 1.5
"""

with open('data/resumes.txt', 'w') as f:
    f.write(content)

print("File 'data/resumes.txt' created successfully!")

File 'data/resumes.txt' created successfully!


In [8]:
with open('data/resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 3 sample résumés

Résumé 1: Alice Johnson — 3 skills, 2.0 years exp

Résumé 2: Bob Smith — 3 skills, 4.5 years exp

Résumé 3: Charlie Davis — 3 skills, 1.5 years exp

=== Full first result ===
{
  "name": "Alice Johnson",
  "email": "alice.j@example.com",
  "phone": "555-0101",
  "education": [
    {
      "degree": "BS in Computer Science",
      "institution": "State University",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "SQL",
    "Machine Learning"
  ],
  "projects": [
    "Built an automated data pipeline for sentiment analysis"
  ],
  "experience_years": 2.0
}
